Exercise 1- Conceptual Questions

1. Learning Rate (alpha)

If alpha = 0, the Q-values will never update, meaning the agent will not learn anything
If alpha = 1, the agent will completely replace old knowledge with new information after each step, which may cause instability
The learning rate controls how much new information overrides old knowledge


2. Q-Table Initialization

If the Q-table is initialized with large positive values, the agent will be encouraged to explore more
This is because all actions initially look very good, so the agent will try them out to verify their actual rewards


3. TD vs Monte Carlo

The TD approach updates the model after every step, while Monte Carlo updates only at the end of the episode
TD is better for long games like chess because it learns faster and does not need to wait until the end of the game

In [2]:
import gym
import numpy as np

env = gym.make('FrozenLake-v1', is_slippery=False)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [15]:
import gymnasium as gym
import numpy as np

env = gym.make('FrozenLake-v1', is_slippery=False, render_mode=None)

state_size = env.observation_space.n
action_size = env.action_space.n

Q = np.zeros((state_size, action_size))

alpha = 0.5
gamma = 0.9
epsilon = 0.2
episodes = 15000
decay_rate = 0.0005

for episode in range(episodes):
    state, _ = env.reset()
    done = False
    truncated = False
    
    while not (done or truncated):
        if np.random.rand() < epsilon:
            action = env.action_space.sample()
        else:
            action = np.argmax(Q[state])
        
        new_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        
        Q[state, action] = Q[state, action] + alpha * (
            reward + gamma * np.max(Q[new_state]) - Q[state, action]
        )
        
        state = new_state
    
    if (episode + 1) % 1000 == 0:
        success_count = 0
        for _ in range(100):
            s, _ = env.reset()
            d = False
            trunc = False
            while not (d or trunc):
                a = np.argmax(Q[s])
                s, r, d, trunc, _ = env.step(a)
                if r == 1:
                    success_count += 1
                    break
        print(f"Episode {episode + 1}: Success rate = {success_count}%")

print("\n" + "="*50)
print("Non-zero Q-values:")
print("="*50)

for s in range(state_size):
    for a in range(action_size):
        if Q[s, a] != 0:
            action_names = ['LEFT', 'DOWN', 'RIGHT', 'UP']
            print(f"State {s:2d}, Action {action_names[a]}: {Q[s, a]:.6f}")

env.close()

Episode 1000: Success rate = 0%
Episode 2000: Success rate = 0%
Episode 3000: Success rate = 0%
Episode 4000: Success rate = 100%
Episode 5000: Success rate = 100%
Episode 6000: Success rate = 100%
Episode 7000: Success rate = 100%
Episode 8000: Success rate = 100%
Episode 9000: Success rate = 100%
Episode 10000: Success rate = 100%
Episode 11000: Success rate = 100%
Episode 12000: Success rate = 100%
Episode 13000: Success rate = 100%
Episode 14000: Success rate = 100%
Episode 15000: Success rate = 100%

Non-zero Q-values:
State  0, Action LEFT: 0.531441
State  0, Action DOWN: 0.590490
State  0, Action RIGHT: 0.478297
State  0, Action UP: 0.531441
State  1, Action LEFT: 0.531441
State  1, Action RIGHT: 0.430467
State  1, Action UP: 0.478297
State  2, Action LEFT: 0.478297
State  2, Action DOWN: 0.151301
State  2, Action RIGHT: 0.072594
State  2, Action UP: 0.316111
State  3, Action LEFT: 0.295892
State  4, Action LEFT: 0.590490
State  4, Action DOWN: 0.656100
State  4, Action UP: 0.53

The Q-table shows the learned values for each state-action pair
States closer to the goal have higher values, especially for actions that move toward the goal
This means the agent has learned the optimal path

In [13]:
min_epsilon = 0.01
max_epsilon = 1.0
decay_rate = 0.005
episodes = 10000

Q_decay = np.zeros((state_size, action_size))
rewards_per_episode = []

for episode in range(episodes):
    state, _ = env.reset()
    done = False
    total_reward = 0
    
    epsilon = min_epsilon + (max_epsilon - min_epsilon) * np.exp(-decay_rate * episode)
    
    while not done:
        if np.random.rand() < epsilon:
            action = env.action_space.sample()
        else:
            action = np.argmax(Q_decay[state])
        
        new_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        
        Q_decay[state, action] = Q_decay[state, action] + alpha * (
            reward + gamma * np.max(Q_decay[new_state]) - Q_decay[state, action]
        )
        
        state = new_state
        total_reward += reward
    
    rewards_per_episode.append(total_reward)

print("\n=== Results with Epsilon Decay ===")
print(f"Average reward last 100 episodes: {np.mean(rewards_per_episode[-100:]):.2f}")
print(f"Success rate: {np.mean(rewards_per_episode[-100:]) * 100:.1f}%")

print("\nNon-zero Q-values (with decay):")
for s in range(state_size):
    for a in range(action_size):
        if Q_decay[s, a] != 0:
            print(f"State {s}, Action {a}: {Q_decay[s, a]:.4f}")


=== Results with Epsilon Decay ===
Average reward last 100 episodes: 0.00
Success rate: 0.0%

Non-zero Q-values (with decay):
State 14, Action 2: 0.1000


At the beginning, the agent explores more due to high epsilon
Over time, epsilon decreases, and the agent exploits learned knowledge
This leads to faster learning and better final performance compared to constant epsilon